In [2]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
from matplotlib.colors import LogNorm
import re

In [ ]:
archivo_x = "perfil_flujo_x_ycte.out"
archivo_y = "perfil_flujo_y_xcte.out"

x_c = 0.0
y_c = 0.0

a = 1.6
b = 1.0

y_linea = 0.0
x_linea = 0.0

def leer_perfil_1d(archivo):
    datos = []
    leyendo = False

    with open(archivo, "r", encoding="utf-8", errors="ignore") as f:
        for linea in f:
            if re.search(r"#\s*\w.*lower.*upper", linea.lower()):
                leyendo = True
                continue

            if leyendo:
                if linea.strip() == "" or linea.strip().startswith("#") or "sum over" in linea.lower():
                    if len(datos) > 0:
                        break
                    continue

                partes = linea.split()

                try:
                    nums = [float(p.replace("D", "E")) for p in partes]
                except ValueError:
                    continue

                if len(nums) >= 4:
                    datos.append(nums)

    datos = np.array(datos)

    bordes_min = datos[:, 0]
    bordes_max = datos[:, 1]
    centro = 0.5 * (bordes_min + bordes_max)
    valor = datos[:, 2]
    err_rel = datos[:, 3]

    return centro, valor, err_rel

x, flujo_x, err_x = leer_perfil_1d(archivo_x)
y, flujo_y, err_y = leer_perfil_1d(archivo_y)

if abs(y_linea - y_c) <= b:
    dx = a * np.sqrt(1 - ((y_linea - y_c) / b) ** 2)
    x_pouch_min = x_c - dx
    x_pouch_max = x_c + dx
else:
    x_pouch_min = np.nan
    x_pouch_max = np.nan

if abs(x_linea - x_c) <= a:
    dy = b * np.sqrt(1 - ((x_linea - x_c) / a) ** 2)
    y_pouch_min = y_c - dy
    y_pouch_max = y_c + dy
else:
    y_pouch_min = np.nan
    y_pouch_max = np.nan

plt.figure(figsize=(7, 5))
plt.plot(x, flujo_x, marker="o", markersize=3)
plt.yscale("log")

if not np.isnan(x_pouch_min):
    plt.axvspan(x_pouch_min, x_pouch_max, alpha=0.25, label="Pouch")

plt.xlabel("x [cm]")
plt.ylabel("Flujo [1/cm²/source]")
plt.title(f"Perfil horizontal, y = {y_linea:.2f} cm")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(y, flujo_y, marker="o", markersize=3)
plt.yscale("log")

if not np.isnan(y_pouch_min):
    plt.axvspan(y_pouch_min, y_pouch_max, alpha=0.25, label="Pouch")

plt.xlabel("y [cm]")
plt.ylabel("Flujo [1/cm²/source]")
plt.title(f"Perfil vertical, x = {x_linea:.2f} cm")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# con el elipsoide

archivo_x = "perfil_flujo_x_ycte.out"
archivo_y = "perfil_flujo_y_xcte.out"

x_c = 0.0
y_c = 0.0

a = 1.6
b = 1.0

x_linea = 0.0
y_linea = 0.0

def leer_perfil_1d(archivo):
    datos = []
    leyendo = False

    with open(archivo, "r", encoding="utf-8", errors="ignore") as f:
        for linea in f:
            l = linea.lower()

            if "lower" in l and "upper" in l and "#" in l:
                leyendo = True
                continue

            if leyendo:
                if "sum over" in l:
                    break

                if linea.strip() == "" or linea.strip().startswith("#"):
                    continue

                partes = linea.split()

                try:
                    nums = [float(p.replace("D", "E")) for p in partes]
                except ValueError:
                    continue

                if len(nums) >= 4:
                    datos.append(nums)

    datos = np.array(datos)

    if datos.size == 0:
        raise ValueError(f"No pude leer datos numéricos en {archivo}")

    minimo = datos[:, 0]
    maximo = datos[:, 1]
    centro = 0.5 * (minimo + maximo)
    flujo = datos[:, 2]
    err_rel = datos[:, 3]

    return centro, flujo, err_rel

x, flujo_x, err_x = leer_perfil_1d(archivo_x)
y, flujo_y, err_y = leer_perfil_1d(archivo_y)

flujos_positivos = np.concatenate([flujo_x[flujo_x > 0], flujo_y[flujo_y > 0]])

vmin = np.min(flujos_positivos)
vmax = np.max(flujos_positivos)

fig, ax = plt.subplots(figsize=(8, 6))

pouch = Ellipse(
    (x_c, y_c),
    width=2*a,
    height=2*b,
    fill=False,
    linewidth=2,
    label="Pouch"
)

ax.add_patch(pouch)

sc1 = ax.scatter(
    x,
    np.full_like(x, y_linea),
    c=flujo_x,
    norm=LogNorm(vmin=vmin, vmax=vmax),
    s=45,
    label=f"Perfil x, y = {y_linea:.2f} cm"
)

sc2 = ax.scatter(
    np.full_like(y, x_linea),
    y,
    c=flujo_y,
    norm=LogNorm(vmin=vmin, vmax=vmax),
    s=45,
    label=f"Perfil y, x = {x_linea:.2f} cm"
)

ax.plot(x, np.full_like(x, y_linea), linewidth=1)
ax.plot(np.full_like(y, x_linea), y, linewidth=1)

ax.set_aspect("equal")
ax.set_xlabel("x [cm]")
ax.set_ylabel("y [cm]")
ax.set_title("Perfiles de flujo sobre la geometría del pouch")
ax.legend()

margen = 0.4
ax.set_xlim(x_c - a - margen, x_c + a + margen)
ax.set_ylim(y_c - b - margen, y_c + b + margen)

cbar = plt.colorbar(sc1, ax=ax)
cbar.set_label("Flujo [1/cm²/source]")

plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(x, flujo_x, marker="o", markersize=3)
plt.yscale("log")
plt.axvspan(x_c - a, x_c + a, alpha=0.2, label="Pouch")
plt.xlabel("x [cm]")
plt.ylabel("Flujo [1/cm²/source]")
plt.title(f"Perfil horizontal: y = {y_linea:.2f} cm")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 5))
plt.plot(y, flujo_y, marker="o", markersize=3)
plt.yscale("log")
plt.axvspan(y_c - b, y_c + b, alpha=0.2, label="Pouch")
plt.xlabel("y [cm]")
plt.ylabel("Flujo [1/cm²/source]")
plt.title(f"Perfil vertical: x = {x_linea:.2f} cm")
plt.legend()
plt.tight_layout()
plt.show()